# Validation

Charts for the four validation questions. Every number comes from
`courtvision.validation`, the same code behind `courtvision validate` and
[`docs/validation.md`](../docs/validation.md), so the two never disagree.

Run `uv run courtvision metrics build` first.

In [ ]:
import duckdb
import plotly.express as px
import plotly.io as pio

from courtvision import validation as V
from courtvision.config import get_paths

pio.renderers.default = "notebook_connected"
con = duckdb.connect(str(get_paths().duckdb), read_only=True)

## 1. Stabilization

Split-half reliability of each player-season measure. `k` is the exposure at
which a measure is half signal; it is also the shrinkage constant.

In [ ]:
stab = V.stabilization(con)
fig = px.bar(stab[stab.tier == "A"].sort_values("k"), x="k", y="measure", color="family",
             orientation="h", log_x=True, title="Exposure to reach reliability 0.5 (tier A)")
fig.show()
stab

## 2. Persistence

First-half vs second-half correlation of team measures. Shot quality (where
shots come from) should persist far more than shot making (whether they go in).

In [ ]:
halves = V._team_halves(con)
pers = V.persistence(halves)
px.bar(pers, x="r_first_second", y="measure", orientation="h",
       title="Team first-half vs second-half correlation").show()
pers

## 3. Signal beyond Four Factors and net rating

Out-of-sample (leave-one-season-out) R² for predicting second-half net rating.

In [ ]:
pred = V.predictive(halves)
px.bar(pred, x="cv_r2", y="model", orientation="h",
       title=f"Predicting second-half net rating ({pred.attrs['team_seasons']} team-seasons)").show()
pred

## 4. Tier agreement (T7)

How closely box-score tiers track the play-by-play tier where both exist.

In [ ]:
tiers = V.tier_agreement(con)
px.bar(tiers, x="r_value", y="measure", color="other_tier", facet_col="level",
       orientation="h", barmode="group", title="Tier A vs box-score tiers").show()
tiers

## 5. Scale (T5)

In [ ]:
V.scale_summary(con)